# Testing NomNom MCP Tools

**Phase 6 — MCP Servers & Claude-as-a-Tool**

This notebook is an interactive harness for exercising the tools exposed by the
NomNom MCP server (`nomnom_mcp_server.py`) **without** needing Claude Code or any
other MCP client in the loop. We speak the MCP protocol (JSON-RPC 2.0 over stdio)
directly from Python so we can:

1. Start the server as a subprocess and complete the MCP handshake.
2. Ask it `tools/list` — *what can you do?*
3. Call each tool with `tools/call` and inspect the structured result.
4. Verify error paths (unknown tool, missing image) behave gracefully.
5. Get a one-glance PASS/FAIL summary.

The three tools under test:

| Tool | Purpose |
|---|---|
| `recommend_meal` | Recommend a meal for a calorie + diet target (real 5-step workflow). |
| `analyze_food_image` | Identify food from an image and estimate macros (Claude vision). |
| `lookup_nutrition` | Query the nutrition knowledge base (RAG-backed). |

> **Why test over stdio directly?** It isolates the *server* from the *client*. If a
> tool misbehaves, we know it's our server logic — not Claude, not the transport.
> This is the same separation the Day 2 skeleton established (protocol first, logic second).

## 1. Configuration

Pick which server file to test against:

- **`nomnom_mcp_server_test.py`** — mock data, no API key or DB needed. Fast, deterministic.
  Best for verifying the *protocol plumbing* and this notebook itself.
- **`nomnom_mcp_server.py`** — the real server. Imports the NomNom backend and calls Claude,
  so it needs `ANTHROPIC_API_KEY` (via `src.config.settings`) and the backend importable.

Start with the mock server. Flip `SERVER_FILE` to the real one once the plumbing is green.

In [ ]:
import sys
from pathlib import Path

# This notebook lives in learning_lab/phase_6/, next to the server files.
PHASE_6_DIR = Path.cwd()
if PHASE_6_DIR.name != "phase_6":
    # Fallback if the notebook is launched from the repo root.
    candidate = PHASE_6_DIR / "learning_lab" / "phase_6"
    if candidate.exists():
        PHASE_6_DIR = candidate

# Choose the server under test.
#   "nomnom_mcp_server_test.py" -> mock data (no API key / DB needed)
#   "nomnom_mcp_server.py"      -> real backend (needs ANTHROPIC_API_KEY)
SERVER_FILE = "nomnom_mcp_server_test.py"
SERVER_PATH = PHASE_6_DIR / SERVER_FILE

# Python interpreter used to launch the server. Default to the one running this
# notebook so the `mcp` package (and, for the real server, the backend deps) resolve.
PYTHON_EXE = sys.executable

assert SERVER_PATH.exists(), f"Server file not found: {SERVER_PATH}"
print(f"Server under test : {SERVER_PATH}")
print(f"Python executable : {PYTHON_EXE}")

## 2. A tiny MCP test client

The MCP handshake is small but strict. The lifecycle for one session is:

```
client --> initialize            ("hello, here's who I am")
server --> result                ("hi, here are my capabilities")
client --> notifications/initialized   ("great, I'm ready")   <- easy to forget!
client --> tools/list            ("what can you do?")
server --> result                (list of tool schemas)
client --> tools/call            ("run recommend_meal with these args")
server --> result                (content[].text holds the JSON payload)
```

We wrap that in a context manager so each `with MCPTestClient(...) as mcp:` block
gets a fresh server process that is always cleaned up — no leaked subprocesses even
if a cell raises.

**Design note:** tool *results* come back as `result.content[0].text`, which is a JSON
**string**. Our server functions return dicts, so we `json.loads` that text back into a
dict for convenient assertions.

In [ ]:
import json
import subprocess
from contextlib import contextmanager


class MCPError(RuntimeError):
    """Raised when the server returns a JSON-RPC error or an unexpected frame."""


class MCPTestClient:
    """Minimal JSON-RPC 2.0 client for an MCP server over stdio."""

    def __init__(self, server_path, python_exe, timeout=60):
        self.server_path = str(server_path)
        self.python_exe = python_exe
        self.timeout = timeout
        self.proc = None
        self._id = 0

    # -- lifecycle ---------------------------------------------------------
    def start(self):
        self.proc = subprocess.Popen(
            [self.python_exe, self.server_path],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1,  # line-buffered: one JSON frame per line
            cwd=str(Path(self.server_path).parent),
        )
        self._handshake()
        return self

    def stop(self):
        if self.proc is None:
            return
        try:
            self.proc.terminate()
            self.proc.wait(timeout=5)
        except Exception:
            self.proc.kill()
        finally:
            self.proc = None

    # -- low-level JSON-RPC ------------------------------------------------
    def _next_id(self):
        self._id += 1
        return self._id

    def _send(self, payload):
        self.proc.stdin.write(json.dumps(payload) + "\n")
        self.proc.stdin.flush()

    def _read(self):
        line = self.proc.stdout.readline()
        if not line:
            stderr = self.proc.stderr.read() if self.proc.stderr else ""
            raise MCPError(f"Server closed the connection. stderr:\n{stderr}")
        return json.loads(line)

    def _request(self, method, params=None):
        req_id = self._next_id()
        self._send({"jsonrpc": "2.0", "id": req_id, "method": method,
                    "params": params or {}})
        resp = self._read()
        if resp.get("error"):
            raise MCPError(f"{method} -> {resp['error']}")
        return resp.get("result", {})

    def _notify(self, method, params=None):
        # Notifications have no id and expect no response.
        self._send({"jsonrpc": "2.0", "method": method, "params": params or {}})

    # -- MCP protocol ------------------------------------------------------
    def _handshake(self):
        self._request("initialize", {
            "protocolVersion": "2024-11-05",
            "capabilities": {},
            "clientInfo": {"name": "notebook-test-client", "version": "1.0"},
        })
        # Required by the spec before issuing further requests.
        self._notify("notifications/initialized")

    def list_tools(self):
        return self._request("tools/list").get("tools", [])

    def call_tool(self, name, arguments):
        """Call a tool and return its payload decoded back into a dict."""
        result = self._request("tools/call",
                                {"name": name, "arguments": arguments})
        content = result.get("content", [])
        if not content or "text" not in content[0]:
            raise MCPError(f"Unexpected tool result shape: {result}")
        text = content[0]["text"]
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return {"_raw_text": text}


@contextmanager
def mcp_session(server_path=None, python_exe=None, timeout=60):
    """Context manager that yields a started client and always cleans up."""
    client = MCPTestClient(server_path or SERVER_PATH,
                           python_exe or PYTHON_EXE, timeout=timeout)
    try:
        yield client.start()
    finally:
        client.stop()


print("MCPTestClient ready.")

## 3. Smoke test — handshake + tool discovery

Before calling anything, confirm the server starts, completes the handshake, and
advertises the three tools we expect. If this cell fails, nothing downstream will work —
fix the plumbing here first.

In [ ]:
EXPECTED_TOOLS = {"recommend_meal", "analyze_food_image", "lookup_nutrition"}

with mcp_session() as mcp:
    tools = mcp.list_tools()

names = {t["name"] for t in tools}
print(f"Server advertised {len(tools)} tool(s):\n")
for t in tools:
    desc = (t.get('description') or '').strip().splitlines()[0] if t.get('description') else ''
    print(f"  - {t['name']}: {desc}")

missing = EXPECTED_TOOLS - names
extra = names - EXPECTED_TOOLS
assert not missing, f"Missing expected tools: {missing}"
print(f"\n[OK] All expected tools present." + (f" (extra: {extra})" if extra else ""))

## 4. Test `recommend_meal`

The headline tool. Give it a calorie target and a diet type; expect a structured meal
recommendation back. We assert on the *shape* (keys present, types sane) rather than the
exact meal name, since the real server's output is non-deterministic (it's an LLM workflow).

In [ ]:
with mcp_session() as mcp:
    result = mcp.call_tool("recommend_meal",
                           {"calories": 600, "diet_type": "vegetarian"})

print(json.dumps(result, indent=2))

assert "error" not in result, f"Tool returned an error: {result.get('error')}"
assert result.get("meal_name"), "Expected a non-empty meal_name"
assert result.get("diet_type") == "vegetarian"
for macro in ("protein_g", "carbs_g", "fat_g"):
    assert isinstance(result.get(macro), (int, float)), f"{macro} should be numeric"
print(f"\n[OK] recommend_meal -> {result['meal_name']} ({result.get('calories')} kcal)")

### Parameterized sweep across diet types

Reuse a single server session to call the tool several times — handy for eyeballing how
the recommendation changes with the inputs, and cheaper than respawning the process each time.

In [ ]:
cases = [
    {"calories": 400, "diet_type": "vegetarian"},
    {"calories": 600, "diet_type": "vegan"},
    {"calories": 800, "diet_type": "omnivore"},
]

with mcp_session() as mcp:
    for args in cases:
        r = mcp.call_tool("recommend_meal", args)
        meal = r.get("meal_name") or r.get("error")
        print(f"  {args['diet_type']:11s} @ {args['calories']:>4} kcal -> {meal}")

## 5. Test `lookup_nutrition`

Query the nutrition knowledge base. Expect a `results` list and, importantly, **citations** —
the whole point of the RAG layer is grounded, attributable answers.

In [ ]:
with mcp_session() as mcp:
    result = mcp.call_tool("lookup_nutrition",
                           {"query": "high protein vegetarian meals"})

print(json.dumps(result, indent=2))

assert "error" not in result, f"Tool returned an error: {result.get('error')}"
assert isinstance(result.get("results"), list) and result["results"], "Expected non-empty results"
assert result.get("citations"), "Expected citations for a RAG-backed answer"
print(f"\n[OK] lookup_nutrition -> {result.get('count')} result(s) with citations")

## 6. Test `analyze_food_image`

This tool needs a real image file on disk. We handle two outcomes:

- **Image found** -> expect `food_name` + estimated macros.
- **Image missing** -> expect a *graceful* structured error (`error` set, `food_name` null),
  **not** a crashed server. Verifying the sad path is as valuable as the happy path.

Set `IMAGE_PATH` to a real photo to exercise the happy path (real server = Claude vision).

In [ ]:
# Point this at a real food photo to test the happy path. Leave as-is to test the
# graceful-error path (file intentionally does not exist).
IMAGE_PATH = "/tmp/nonexistent_food.jpg"

image_exists = Path(IMAGE_PATH).exists()
print(f"IMAGE_PATH = {IMAGE_PATH}  (exists={image_exists})\n")

with mcp_session() as mcp:
    result = mcp.call_tool("analyze_food_image", {"image_path": IMAGE_PATH})

print(json.dumps(result, indent=2))

if image_exists:
    assert "error" not in result, f"Tool returned an error: {result.get('error')}"
    assert result.get("food_name"), "Expected a food_name for a valid image"
    print(f"\n[OK] analyze_food_image -> {result['food_name']}")
else:
    # Mock server ignores the path and returns data; real server reports a clean error.
    handled = ("error" in result) or bool(result.get("food_name"))
    assert handled, "Tool should return a structured response, not crash"
    print("\n[OK] analyze_food_image handled the input gracefully")

## 7. Error handling — unknown tool

A well-behaved server should reject an unknown tool with a JSON-RPC error rather than
hanging or dying. Our client surfaces that as an `MCPError`, which we assert on.

In [ ]:
raised = False
try:
    with mcp_session() as mcp:
        mcp.call_tool("this_tool_does_not_exist", {})
except MCPError as e:
    raised = True
    print(f"Server rejected unknown tool as expected:\n  {e}")

assert raised, "Expected an MCPError for an unknown tool"
print("\n[OK] unknown-tool error path works")

## 8. Full suite — one-glance summary

Run everything together and print a compact PASS/FAIL table. This is the cell to re-run
after editing the server: green across the board means the tools and protocol are healthy.

In [ ]:
def run_suite():
    results = {}

    # 1) discovery
    try:
        with mcp_session() as mcp:
            names = {t["name"] for t in mcp.list_tools()}
        results["list_tools"] = EXPECTED_TOOLS.issubset(names)
    except Exception as e:
        results["list_tools"] = f"ERROR: {e}"

    # 2) recommend_meal
    try:
        with mcp_session() as mcp:
            r = mcp.call_tool("recommend_meal", {"calories": 600, "diet_type": "vegetarian"})
        results["recommend_meal"] = bool(r.get("meal_name")) and "error" not in r
    except Exception as e:
        results["recommend_meal"] = f"ERROR: {e}"

    # 3) lookup_nutrition
    try:
        with mcp_session() as mcp:
            r = mcp.call_tool("lookup_nutrition", {"query": "high protein vegetarian meals"})
        results["lookup_nutrition"] = bool(r.get("results")) and "error" not in r
    except Exception as e:
        results["lookup_nutrition"] = f"ERROR: {e}"

    # 4) analyze_food_image (graceful handling of missing file)
    try:
        with mcp_session() as mcp:
            r = mcp.call_tool("analyze_food_image", {"image_path": "/tmp/nonexistent_food.jpg"})
        results["analyze_food_image"] = ("error" in r) or bool(r.get("food_name"))
    except Exception as e:
        results["analyze_food_image"] = f"ERROR: {e}"

    # 5) unknown-tool error path
    try:
        with mcp_session() as mcp:
            mcp.call_tool("this_tool_does_not_exist", {})
        results["unknown_tool_rejected"] = False  # should have raised
    except MCPError:
        results["unknown_tool_rejected"] = True
    except Exception as e:
        results["unknown_tool_rejected"] = f"ERROR: {e}"

    return results


results = run_suite()

print("=" * 60)
print(f"NomNom MCP Tool Tests  ({SERVER_FILE})")
print("=" * 60)
passed = 0
for name, outcome in results.items():
    ok = outcome is True
    passed += ok
    label = "PASS" if ok else "FAIL"
    detail = "" if isinstance(outcome, bool) else f"  ({outcome})"
    print(f"  [{label}] {name}{detail}")
print("-" * 60)
print(f"  {passed}/{len(results)} checks passed")
print("=" * 60)

## 9. Next steps

- **Flip to the real server:** set `SERVER_FILE = "nomnom_mcp_server.py"` in section 1,
  ensure `ANTHROPIC_API_KEY` is set (the server reads it via `src.config.settings`), and
  re-run section 8. Expect the same checks to pass — same tool contracts, real implementations.
- **Add a real image:** drop a food photo somewhere and point `IMAGE_PATH` at it to exercise
  Claude vision end-to-end in section 6.
- **From here to a real client:** once these tool contracts are green over raw stdio, wiring
  the server into Claude Code is just a config entry — see `04_claude_code_integration.md`.

> **Takeaway:** the mock server and the real server share one interface. Testing the contract
> over stdio — independent of any client — is what lets us swap the implementation underneath
> with confidence.